# V3 Universal Football Model — Architecture & Loss Function

This notebook defines the neural network architecture for Version 3. 
It implements the two major upgrades we discussed:
1. **League Embeddings:** A dedicated embedding layer so the model can learn the specific "meta" of different leagues.
2. **Focal Loss:** Replacing static class weights with a dynamic loss function that forces the model to focus on hard-to-predict examples (like draws and upsets) without brute-forcing an over-prediction.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np


## 1. The Universal Football Net

We separate the input into two parts:
- `x_cont`: The continuous features (Minute, XI values, goal diff, red cards, etc.)
- `x_league`: The categorical league ID, which gets passed through an embedding layer.

In [3]:
class UniversalFootballNet(nn.Module):
    def __init__(self, n_continuous=10, num_leagues=100, embed_dim=4, h1=64, h2=32, dropout=0.3):
        super().__init__()
        
        # 1. League Embedding Layer
        # Transforms a sparse league ID (e.g., 39 for Premier League) into a dense 4D vector
        self.league_embed = nn.Embedding(num_embeddings=num_leagues, embedding_dim=embed_dim)
        
        # Total input to the first hidden layer
        input_dim = n_continuous + embed_dim
        
        # 2. Wider Neural Network Architecture (64 -> 32 -> 3)
        self.fc1 = nn.Linear(input_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        
        # 3-class output for standard 90-minute regulation (Away, Draw, Home)
        self.head = nn.Linear(h2, 3)
        
        self.drop = nn.Dropout(dropout)
        self.act = nn.ReLU()
        
    def forward(self, x_cont, x_league):
        # Embed the league categorical variable
        league_vec = self.league_embed(x_league)
        
        # Concatenate continuous features with the league embedding
        x = torch.cat([x_cont, league_vec], dim=1)
        
        # Forward pass through hidden layers
        x = self.drop(self.act(self.bn1(self.fc1(x))))
        x = self.drop(self.act(self.bn2(self.fc2(x))))
        
        # Logits output (no softmax here, handled by loss function during training)
        logits = self.head(x)
        return logits


## 2. Focal Loss implementation

Standard CrossEntropyLoss treats all errors equally. If the model predicts "Home Win" and the true result is a Draw, it gets a standard penalty. 

**Focal Loss** adds a focusing parameter $\gamma$. If the model is already highly confident and correct, the loss drops to near zero. If the model is completely wrong (which happens frequently with draws), the loss is magnified. This forces the network to spend its gradients learning the complex boundaries around Draws, rather than just optimizing the easy "Man City goes up 3-0" predictions.

In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        alpha: Tensor of weights per class to handle basic class imbalance.
        gamma: Focusing parameter. Higher gamma = more focus on hard examples.
        """
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha
        
    def forward(self, inputs, targets):
        # Compute standard cross entropy loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        
        # Compute the probability of the true class (pt)
        pt = torch.exp(-ce_loss)
        
        # Apply the focal scaling factor: (1 - pt)^gamma
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


## 3. Dummy Forward Pass Test
We create a small fake batch of data representing 4 snapshots to ensure the tensor shapes align and the model compiles correctly.

In [5]:
# 1. Initialize the Model and Loss
model = UniversalFootballNet(n_continuous=10, num_leagues=50, embed_dim=4)
# Slight alpha bump to the Draw class (Index 1) just to help Focal Loss along
loss_fn = FocalLoss(alpha=torch.tensor([1.0, 1.5, 1.0]), gamma=2.0)

# 2. Create Dummy Data (Batch Size = 4)
# Continuous features: minute, home_xi, away_xi, days_home, days_away, goal_diff, score_state, home_reds, away_reds, is_knockout
dummy_cont = torch.tensor([
    [0.0, 850.0, 400.0, 7.0, 7.0,  0.0, 1.0, 0.0, 0.0, 0.0],  # Min 0, Tied
    [45.0, 850.0, 400.0, 7.0, 7.0, 1.0, 2.0, 0.0, 0.0, 0.0],  # Min 45, Home leading
    [75.0, 850.0, 400.0, 7.0, 7.0, 1.0, 2.0, 0.0, 1.0, 0.0],  # Min 75, Home leading, Away has red
    [90.0, 850.0, 400.0, 7.0, 7.0, 0.0, 1.0, 0.0, 1.0, 0.0],  # Min 90, Away equalized (Tied)
], dtype=torch.float32)

# Categorical feature: League ID (e.g., 39 for Premier League)
dummy_league = torch.tensor([39, 39, 39, 39], dtype=torch.long)

# Target outcomes: 0=Away, 1=Draw, 2=Home
dummy_targets = torch.tensor([1, 2, 2, 1], dtype=torch.long)

# 3. Run Forward Pass
model.eval() # Set to eval to disable dropout/batchnorm for testing
with torch.no_grad():
    logits = model(dummy_cont, dummy_league)
    loss = loss_fn(logits, dummy_targets)

print(f"Continuous Inputs Shape: {dummy_cont.shape}")
print(f"League Inputs Shape:     {dummy_league.shape}")
print(f"Output Logits Shape:     {logits.shape} -> [Away, Draw, Home]")
print(f"Focal Loss Value:        {loss.item():.4f}")

print("\n✅ Model Architecture compiled and data shapes align perfectly.")


Continuous Inputs Shape: torch.Size([4, 10])
League Inputs Shape:     torch.Size([4])
Output Logits Shape:     torch.Size([4, 3]) -> [Away, Draw, Home]
Focal Loss Value:        22.9714

✅ Model Architecture compiled and data shapes align perfectly.
